In [1]:
from pathlib import Path
import subprocess, os

# ==============================
# OPTIONAL: set this if Jupyter is launched outside the repo
# (recommended default for robustness)
# ==============================
REPO_HINT = Path("/Users/mkrishan/Documents/boolearn-main")
# Users can comment this out if they run notebooks from inside the repo


def find_repo_root():
    candidates = []

    # 1) user-provided hint
    if REPO_HINT is not None:
        candidates.append(Path(REPO_HINT).expanduser().resolve())

    # 2) upward search from current working directory
    cwd = Path.cwd().resolve()
    candidates += [cwd] + list(cwd.parents)

    # 3) common fallbacks
    home = Path.home()
    candidates += [
        home / "boolearn-main",
        home / "Documents" / "boolearn-main",
        home / "Desktop" / "boolearn-main",
    ]

    # valid repo = boolearn/src/makefile exists
    for p in candidates:
        if (p / "boolearn" / "src" / "makefile").exists():
            return p

    raise RuntimeError(
        "❌ Could not locate Boolearn repository root.\n"
        "Fix: set REPO_HINT = Path('.../boolearn-main') at the top of this cell.\n"
        f"Current working directory: {cwd}"
    )


# ==============================
# Locate repo + key paths
# ==============================
ROOT = find_repo_root()
SRC  = ROOT / "boolearn" / "src"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("✅ Repo root:", ROOT)
print("📁 SRC:", SRC)
print("📁 DATA:", DATA)
print("📁 RESULTS:", RESULTS)

# ==============================
# Build boolearn binaries
# ==============================
subprocess.run(["make", "all"], cwd=str(SRC), check=True)

# ==============================
# Add boolearn/src to PATH
# ==============================
os.environ["PATH"] = f"{SRC}:{os.environ.get('PATH','')}"
print("✅ PATH updated with boolearn/src")

✅ Repo root: /Users/mkrishan/Documents/boolearn-main
📁 SRC: /Users/mkrishan/Documents/boolearn-main/boolearn/src
📁 DATA: /Users/mkrishan/Documents/boolearn-main/data
📁 RESULTS: /Users/mkrishan/Documents/boolearn-main/results
make: Nothing to be done for `all'.
✅ PATH updated with boolearn/src


In [3]:
from pathlib import Path

LOGIC = DATA / "logic"
dat_path = LOGIC / "m5k1.dat"
assert dat_path.exists(), f"Missing {dat_path}"

N_train = 128  # for m5k1
print("Dataset:", dat_path)
print("N_train:", N_train)

# quick header peek
with open(dat_path) as f:
    for _ in range(6):
        print(f.readline().rstrip())

Dataset: /Users/mkrishan/Documents/boolearn-main/data/logic/m5k1.dat
N_train: 128
10000
2 32
2 32

 1 1 0 0 1 1 1 0 1 1 1 0 1 1 1 1 1 0 1 1 0 1 0 0 1 1 1 0 0 1 0 0
 1 1 1 0 0 0 1 1 1 1 1 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 0 1 0 1 0


In [11]:
import subprocess
from pathlib import Path

LOGIC = DATA / "logic"
RESULTS_DIR = RESULTS / "m_series_sanity"
NET_DIR = RESULTS_DIR / "nets"
NET_DIR.mkdir(parents=True, exist_ok=True)

layered = SRC / "layered"
assert layered.exists(), layered

m5_wth = NET_DIR / "m5.wth"
m5_net = NET_DIR / "m5.net"

with open(m5_wth, "w") as f:
    f.write("5\n32 32 32 32 32 32\n")

print("Wrote:", m5_wth)
print(m5_wth.read_text())

if not m5_net.exists():
    cmd = [str(layered), "0.5", str(m5_wth), str(m5_net)]
    print("Running:", " ".join(cmd))

    res = subprocess.run(cmd, cwd=str(NET_DIR), capture_output=True, text=True, check=False)
    print("layered returncode:", res.returncode)
    if res.stdout.strip():
        print("\n--- layered stdout (tail) ---")
        print("\n".join(res.stdout.splitlines()[-20:]))
    if res.stderr.strip():
        print("\n--- layered stderr (tail) ---")
        print("\n".join(res.stderr.splitlines()[-20:]))

else:
    print("Found existing:", m5_net)

# treat existence as success
assert m5_net.exists() and m5_net.stat().st_size > 0, f"m5.net not produced: {m5_net}"
print("✅ Using net:", m5_net, "| bytes:", m5_net.stat().st_size)

# optional: show header line for sanity
with open(m5_net) as f:
    print("Header:", f.readline().strip())

Wrote: /Users/mkrishan/Documents/boolearn-main/results/m_series_sanity/nets/m5.wth
5
32 32 32 32 32 32

Found existing: /Users/mkrishan/Documents/boolearn-main/results/m_series_sanity/nets/m5.net
✅ Using net: /Users/mkrishan/Documents/boolearn-main/results/m_series_sanity/nets/m5.net | bytes: 132018
Header: 193  33  32  5280


In [13]:
import subprocess, time, re, select
from pathlib import Path

train_exe = SRC / "train"
assert train_exe.exists()

LOGIC = DATA / "logic"
dat_path = LOGIC / "m5k1.dat"
assert dat_path.exists()

N_train = 128
run_id = "m5k1B"   # short id, avoids boolearn buffer overflow

RUN_DIR = RESULTS / "m_series_sanity"
gap_path = RUN_DIR / f"{run_id}.gap"

# fast sanity hyperparams
btfn   = "3.0"
beta   = "0.1"
gam    = "1e-4"
epochs = "60"        # more frequent gap writes
itermax= "20000"     # sanity
gapstop= "0.01"
trials = "1"

cmd = [
    str(train_exe),
    str(m5_net),            # <-- the generated net
    str(dat_path),
    str(N_train),
    btfn, beta, gam, epochs, itermax, gapstop, trials,
    run_id
]

print("Running boolearn sanity:")
print(" ".join(cmd))
print("Writing to:", RUN_DIR)

proc = subprocess.Popen(
    cmd,
    cwd=str(RUN_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

last_gap_lines = 0
printed_header = False

try:
    while True:
        # non-blocking stdout discard
        if proc.stdout is not None:
            r, _, _ = select.select([proc.stdout], [], [], 0.0)
            if r:
                _ = proc.stdout.readline()

        if gap_path.exists():
            lines = [ln.strip() for ln in gap_path.read_text().splitlines() if ln.strip()]
            if lines and not printed_header:
                print("\niter\ttrain%\ttest%")
                printed_header = True

            if len(lines) > last_gap_lines:
                for ln in lines[last_gap_lines:]:
                    it = ln.split()[0]
                    pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", ln)
                    if len(pcts) >= 2:
                        tr, te = float(pcts[-2]), float(pcts[-1])
                        print(f"{it}\t{tr:.3f}\t{te:.3f}")
                last_gap_lines = len(lines)

        if proc.poll() is not None:
            break

        time.sleep(0.2)

finally:
    try: proc.kill()
    except Exception: pass

print("\nReturn code:", proc.returncode)
print("Artifacts:")
for p in sorted(RUN_DIR.glob(run_id + ".*")):
    print(" -", p.name, "|", p.stat().st_size, "bytes")

Running boolearn sanity:
/Users/mkrishan/Documents/boolearn-main/boolearn/src/train /Users/mkrishan/Documents/boolearn-main/results/m_series_sanity/nets/m5.net /Users/mkrishan/Documents/boolearn-main/data/logic/m5k1.dat 128 3.0 0.1 1e-4 60 20000 0.01 1 m5k1B
Writing to: /Users/mkrishan/Documents/boolearn-main/results/m_series_sanity

iter	train%	test%
2	48.999	49.412
3	49.023	49.340
4	49.243	49.183
5	48.657	49.287
6	49.268	49.172
7	49.780	49.243
8	49.072	49.153
9	49.023	49.228
10	48.926	49.286
11	49.390	49.240
12	49.219	49.247
13	49.268	49.242
14	49.854	49.270
15	48.901	49.243
16	49.780	49.193
17	48.877	49.233
18	49.365	49.190
20	49.194	49.188
24	48.926	49.183
28	48.804	49.135
33	48.828	49.145
38	49.561	49.151
45	49.097	49.138
53	48.755	49.110
62	49.585	49.010
74	49.146	48.994
87	48.413	48.873
102	48.584	48.798
120	48.364	48.763
142	49.902	48.751
167	47.876	48.919
197	49.609	49.172
233	50.171	49.710
274	49.414	50.342
323	51.416	51.383
381	53.906	52.323
450	56.787	53.635
530	59.009	55.3

In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

def log_spaced_steps(max_steps: int, n_points: int = 25):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

# Depth-5 width-32 MLP: 32->32->...->32 (5 affine layers total)
class MLP32(nn.Module):
    def __init__(self, depth=5):
        super().__init__()
        layers = [nn.Linear(32, 32), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Linear(32, 32), nn.ReLU(inplace=True)]
        layers += [nn.Linear(32, 32)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# load same dataset
X_all, Y_all, IN, OUT = load_dat_bits(dat_path)
assert IN == 32 and OUT == 32

Xtr = X_all[:N_train].to(device)
Ytr = Y_all[:N_train].to(device)
Xte = X_all[N_train:].to(device)
Yte = Y_all[N_train:].to(device)

MAX_STEPS = 20000
LOG_STEPS = log_spaced_steps(MAX_STEPS, n_points=25)

model = MLP32(depth=5).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

gd_log = RUN_DIR / "m5k1_gd_adamw_sanity.log"
with open(gd_log, "w") as f:
    f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

def eval_and_log(step):
    model.eval()
    with torch.no_grad():
        tr_logits = model(Xtr); te_logits = model(Xte)
        tr_loss = loss_fn(tr_logits, Ytr).item()
        te_loss = loss_fn(te_logits, Yte).item()
        tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
        te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
    with open(gd_log, "a") as f:
        f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")
    print(f"[GD] step={step:>6d}  train={tr_bit:6.2f}%  test={te_bit:6.2f}%  exact={tr_ex:5.1f}/{te_ex:5.1f}")

eval_and_log(0)
model.train()

for step in range(1, MAX_STEPS + 1):
    opt.zero_grad(set_to_none=True)
    loss = loss_fn(model(Xtr), Ytr)
    loss.backward()
    opt.step()
    if step in LOG_STEPS:
        eval_and_log(step)

print("Saved GD log:", gd_log)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/mkrishan/opt/anaconda3/envs/boolearn/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/mkrishan/opt/anaconda3/envs/boolearn/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/mkrishan/opt/anaconda3/envs/boolearn/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/mkrishan/opt/anaconda3/envs/boolearn/lib/python3.10/site-packages/trait

device: cpu
[GD] step=     0  train= 48.97%  test= 49.69%  exact=  0.0/  0.0
[GD] step=     1  train= 49.54%  test= 49.77%  exact=  0.0/  0.0
[GD] step=     2  train= 49.88%  test= 50.01%  exact=  0.0/  0.0
[GD] step=     3  train= 50.42%  test= 50.14%  exact=  0.0/  0.0
[GD] step=     5  train= 50.68%  test= 50.19%  exact=  0.0/  0.0
[GD] step=     8  train= 50.85%  test= 50.37%  exact=  0.0/  0.0
[GD] step=    12  train= 51.20%  test= 50.25%  exact=  0.0/  0.0
[GD] step=    18  train= 51.42%  test= 50.11%  exact=  0.0/  0.0
[GD] step=    27  train= 53.12%  test= 51.02%  exact=  0.0/  0.0
[GD] step=    41  train= 58.20%  test= 54.24%  exact=  0.0/  0.0
[GD] step=    62  train= 64.84%  test= 58.77%  exact=  0.0/  0.0
[GD] step=    94  train= 71.04%  test= 64.26%  exact=  0.0/  0.0
[GD] step=   141  train= 76.86%  test= 68.52%  exact=  0.0/  0.1
[GD] step=   214  train= 84.16%  test= 71.64%  exact=  1.6/  0.1
[GD] step=   323  train= 90.60%  test= 72.73%  exact=  8.6/  0.1
[GD] step=   